In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [5]:
df = pd.read_csv("RK_Matcon.csv")
df.head()

,Date,SKU_ID,Category,Brand,Supplier,Sales_Channel,Region,Net_Units_Sold,Selling_Price,Revenue,Cost,Gross_Margin,Discount_Percent,Stock_Level,Lead_Time_Days,MOQ,Credit_Days
0,2025-01-01,SKU00001,Adhesives,Nuvoco Vistas Corporation Limited,Allianz Business Corporation,Project,Ethiopia,3,"1,889.90","5,669.71","3,029.24","2,640.47",0,425,36,153,90
1,2025-01-01,SKU00005,Electricals,Nuvoco Vistas Corporation Limited,Hariom Enterprises,Contractor,Benin,2,"2,071.06","4,142.12","3,735.96",406.16,5,549,32,189,30
2,2025-01-01,SKU00007,Adhesives,Somany Ceramics,Pegasus Corporation,Project,Ethiopia,2,"2,284.39","4,568.78","2,842.62","1,726.16",0,215,11,255,60
3,2025-01-01,SKU00009,Cement,PrivateLabel,Hariom Enterprises,Project,Ethiopia,8,"1,100.82","8,806.59","4,469.20","4,337.39",0,414,31,26,0
4,2025-01-01,SKU00011,Cement,Jaquar,Saluja & co.,Retail,Benin,3,881.37,"2,644.12","3,619.14",-975.02,10,449,25,155,0


In [6]:
df.shape

(76660, 17)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76660 entries, 0 to 76659
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Date              76660 non-null  object 
 1   SKU_ID            76660 non-null  object 
 2   Category          76660 non-null  object 
 3   Brand             76660 non-null  object 
 4   Supplier          76660 non-null  object 
 5   Sales_Channel     76660 non-null  object 
 6   Region            76660 non-null  object 
 7   Net_Units_Sold    76660 non-null  int64  
 8   Selling_Price     75099 non-null  float64
 9   Revenue           76660 non-null  float64
 10  Cost              76660 non-null  float64
 11  Gross_Margin      75094 non-null  float64
 12  Discount_Percent  76660 non-null  int64  
 13  Stock_Level       76660 non-null  int64  
 14  Lead_Time_Days    76660 non-null  int64  
 15  MOQ               76660 non-null  int64  
 16  Credit_Days       76660 non-null  int64 

In [8]:
df.isna().sum().sort_values(ascending=False)

Gross_Margin        1566
Selling_Price       1561
Revenue                0
MOQ                    0
Lead_Time_Days         0
Stock_Level            0
Discount_Percent       0
Cost                   0
Date                   0
SKU_ID                 0
Net_Units_Sold         0
Region                 0
Sales_Channel          0
Supplier               0
Brand                  0
Category               0
Credit_Days            0
dtype: int64

In [9]:
df.describe()

,Net_Units_Sold,Selling_Price,Revenue,Cost,Gross_Margin,Discount_Percent,Stock_Level,Lead_Time_Days,MOQ,Credit_Days
count,"76,660.00","75,099.00","76,660.00","76,660.00","75,094.00","76,660.00","76,660.00","76,660.00","76,660.00","76,660.00"
mean,5.37,"1,746.76","9,548.29","6,834.42","2,709.80",4.23,380.43,24.33,154.24,25.33
std,3.04,951.92,"7,877.67","5,813.19","7,310.08",6.16,145.75,12.11,83.75,30.32
min,-4.00,47.98,0.00,0.00,"-40,858.74",0.00,0.00,3.00,10.00,0.00
25%,3.00,911.04,"3,616.20","2,442.38","-1,147.45",0.00,304.00,14.00,79.00,0.00
50%,5.00,"1,705.04","7,516.41","5,275.98","1,884.32",0.00,393.00,25.00,159.00,0.00
75%,7.00,"2,561.33","13,411.28","9,645.22","6,257.30",5.00,476.00,35.00,224.00,30.00
max,24.00,"3,493.82","66,812.63","46,973.30","60,346.59",20.00,898.00,44.00,298.00,90.00


In [10]:
df["Date"] = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.to_period("M")
df["Year"] = df["Date"].dt.year

In [11]:
df["Calculated_GM"] = df["Revenue"] - df["Cost"]
df["GM_Difference"] = df["Gross_Margin"] - df["Calculated_GM"]

df["GM_Difference"].abs().sum()

np.float64(184.96000002814282)

In [13]:
# How frequently do we sell at a loss

neg_margin_df = df[df["Calculated_GM"] < 0]

neg_margin_count = len(neg_margin_df)
total_count = len(df)

neg_margin_pct = (neg_margin_count / total_count) * 100

neg_margin_count, round(neg_margin_pct, 2)

(26822, 34.99)

In [14]:
# Financial impact of the negative margin/ 

total_margin = df["Calculated_GM"].sum()
negative_margin_impact = neg_margin_df["Calculated_GM"].sum()

round(total_margin, 2), round(negative_margin_impact, 2)

(np.float64(208045169.25), np.float64(-108483336.77))

In [17]:
# Transactions where price < unit cost

df["Unit_Cost"] = df["Cost"] / df["Net_Units_Sold"].replace(0, np.nan)
df["Below_Cost_Flag"] = df["Selling_Price"] < df["Unit_Cost"]

below_cost_count = df["Below_Cost_Flag"].sum()
below_cost_pct = (below_cost_count / len(df)) * 100

below_cost_count, round(below_cost_pct, 2)

(np.int64(26646), np.float64(34.76))

In [18]:
# ----Channel-Level Profitability Analysis-----

# Aggregate financials by Sales Channel
channel_summary = df.groupby("Sales_Channel").agg(
    Total_Revenue=("Revenue", "sum"),
    Total_Cost=("Cost", "sum"),
    Total_Gross_Margin=("Calculated_GM", "sum"),
    Transactions=("SKU_ID", "count")
)

# Calculate margin percentage
channel_summary["Gross_Margin_%"] = (
    channel_summary["Total_Gross_Margin"] / channel_summary["Total_Revenue"]
) * 100

channel_summary.sort_values("Total_Gross_Margin")

,Total_Revenue,Total_Cost,Total_Gross_Margin,Transactions,Gross_Margin_%
Sales_Channel,,,,,
Contractor,"242,898,682.53","174,881,187.08","68,017,495.45",25500,28.00
Project,"245,204,129.57","176,080,730.61","69,123,398.96",25708,28.19
Retail,"243,868,718.78","172,964,443.94","70,904,274.84",25452,29.07


In [19]:
# Category-Level Profitability Analysis

category_summary = df.groupby("Category").agg(
    Total_Revenue=("Revenue", "sum"),
    Total_Cost=("Cost", "sum"),
    Total_Gross_Margin=("Calculated_GM", "sum"),
    Transactions=("SKU_ID", "count")
)

category_summary["Gross_Margin_%"] = (
    category_summary["Total_Gross_Margin"] / category_summary["Total_Revenue"]
) * 100

category_summary.sort_values("Gross_Margin_%")

,Total_Revenue,Total_Cost,Total_Gross_Margin,Transactions,Gross_Margin_%
Category,,,,,
Tiles,"104,660,469.09","84,278,818.25","20,381,650.84",10527,19.47
Adhesives,"109,122,523.80","87,117,561.23","22,004,962.57",12484,20.17
Sanitaryware,"106,444,876.20","77,417,823.27","29,027,052.93",12028,27.27
Plumbing,"99,790,439.31","72,509,814.34","27,280,624.97",10134,27.34
CP Fittings,"91,349,557.03","65,237,865.16","26,111,691.87",10073,28.58
Cement,"118,253,455.80","76,876,863.82","41,376,591.98",11922,34.99
Electricals,"102,350,209.65","60,487,615.56","41,862,594.09",9492,40.90


In [21]:
# Negative margins by category

neg_category_summary = df[df["Calculated_GM"] < 0].groupby("Category").agg(
    Negative_Transactions=("SKU_ID", "count"),
    Total_Loss=("Calculated_GM", "sum")
)

neg_category_summary.sort_values("Total_Loss")


,Negative_Transactions,Total_Loss
Category,,
Adhesives,5304,"-20,026,620.64"
Tiles,4148,"-19,531,670.22"
Sanitaryware,4231,"-19,087,057.46"
CP Fittings,3896,"-16,784,687.80"
Cement,3780,"-14,374,485.00"
Plumbing,3423,"-12,263,510.82"
Electricals,2040,"-6,415,304.83"


In [22]:
# SKU-Level Loss Concentration

sku_loss = df[df["Calculated_GM"] < 0].groupby("SKU_ID").agg(
    Total_Loss=("Calculated_GM", "sum"),
    Negative_Transactions=("SKU_ID", "count")
)

# Sort by biggest loss
sku_loss_sorted = sku_loss.sort_values("Total_Loss")

sku_loss_sorted.head(10)

,Total_Loss,Negative_Transactions
SKU_ID,,
SKU00232,"-1,704,019.60",140
SKU00283,"-1,579,173.18",131
SKU00189,"-1,559,741.14",126
SKU00488,"-1,545,606.10",141
SKU00130,"-1,438,206.58",127
SKU00286,"-1,421,814.93",137
SKU00522,"-1,366,874.85",143
SKU00581,"-1,337,517.05",128
SKU00279,"-1,327,032.87",129


In [23]:
# Pareto Analysis on Loss

# Sort SKUs by biggest loss (most negative first)
sku_loss_sorted = sku_loss.sort_values("Total_Loss")

# Calculate cumulative loss
sku_loss_sorted["Cumulative_Loss"] = sku_loss_sorted["Total_Loss"].cumsum()

# Total negative loss
total_negative_loss = sku_loss_sorted["Total_Loss"].sum()

# Cumulative percentage contribution
sku_loss_sorted["Cumulative_%"] = (
    sku_loss_sorted["Cumulative_Loss"] / total_negative_loss
) * 100

sku_loss_sorted.head(20)

,Total_Loss,Negative_Transactions,Cumulative_Loss,Cumulative_%
SKU_ID,,,,
SKU00232,"-1,704,019.60",140,"-1,704,019.60",1.57
SKU00283,"-1,579,173.18",131,"-3,283,192.78",3.03
SKU00189,"-1,559,741.14",126,"-4,842,933.92",4.46
SKU00488,"-1,545,606.10",141,"-6,388,540.02",5.89
SKU00130,"-1,438,206.58",127,"-7,826,746.60",7.21
SKU00286,"-1,421,814.93",137,"-9,248,561.53",8.53
SKU00522,"-1,366,874.85",143,"-10,615,436.38",9.79
SKU00581,"-1,337,517.05",128,"-11,952,953.43",11.02
SKU00279,"-1,327,032.87",129,"-13,279,986.30",12.24


In [24]:
# SKUs causing 50% of loss
sku_loss_sorted[sku_loss_sorted["Cumulative_%"] >= 50].head(1)

# SKUs causing 80% of loss
sku_loss_sorted[sku_loss_sorted["Cumulative_%"] >= 80].head(1)

,Total_Loss,Negative_Transactions,Cumulative_Loss,Cumulative_%
SKU_ID,,,,
SKU00199,"-469,850.38",136,"-86,980,361.39",80.18


In [25]:
# Reset index to get rank position
sku_loss_sorted_reset = sku_loss_sorted.reset_index()

# Find position where cumulative % >= 80
sku_loss_sorted_reset[sku_loss_sorted_reset["Cumulative_%"] >= 80].head(1)

,SKU_ID,Total_Loss,Negative_Transactions,Cumulative_Loss,Cumulative_%
101,SKU00199,"-469,850.38",136,"-86,980,361.39",80.18


In [26]:
# Category concentration within top loss SKUs

top_loss_skus = sku_loss_sorted_reset.iloc[:102]

# Merge with category info
top_loss_skus = top_loss_skus.merge(
    df[["SKU_ID", "Category"]].drop_duplicates(),
    on="SKU_ID",
    how="left"
)

top_loss_skus["Category"].value_counts()

Category
Sanitaryware    19
Tiles           18
Adhesives       17
CP Fittings     16
Cement          14
Plumbing        12
Electricals      6
Name: count, dtype: int64

In [27]:
# Discount comparison: Top loss SKUs vs Others

# Flag top loss SKUs
df["Top_Loss_Flag"] = df["SKU_ID"].isin(top_loss_skus["SKU_ID"])

discount_compare = df.groupby("Top_Loss_Flag").agg(
    Avg_Discount=("Discount_Percent", "mean"),
    Avg_GM=("Calculated_GM", "mean"),
    Transactions=("SKU_ID", "count")
)

discount_compare

,Avg_Discount,Avg_GM,Transactions
Top_Loss_Flag,,,
False,4.21,"4,636.21",63635
True,4.32,"-6,677.95",13025


In [28]:
# Structural Margin Check (MRP vs Cost)

# First calculate theoretical margin if sold at MRP
df["MRP_GM"] = (df["Selling_Price"] / (1 - df["Discount_Percent"]/100)) - df["Unit_Cost"]

# But cleaner approach: approximate MRP using reverse discount
df["Approx_MRP"] = df["Selling_Price"] / (1 - df["Discount_Percent"]/100)

df["MRP_vs_Cost_Margin_%"] = (
    (df["Approx_MRP"] - df["Unit_Cost"]) / df["Approx_MRP"]
) * 100

df.groupby("Top_Loss_Flag")["MRP_vs_Cost_Margin_%"].mean()

Top_Loss_Flag
False     34.90
True    -299.93
Name: MRP_vs_Cost_Margin_%, dtype: float64

In [29]:
# Compare Avg Selling Price vs Avg Unit Cost

price_cost_compare = df.groupby("Top_Loss_Flag").agg(
    Avg_Selling_Price=("Selling_Price", "mean"),
    Avg_Unit_Cost=("Unit_Cost", "mean")
)

price_cost_compare

,Avg_Selling_Price,Avg_Unit_Cost
Top_Loss_Flag,,
False,"1,957.06","1,124.95"
True,718.75,"1,976.18"


In [30]:
# Profit impact if top loss SKUs removed

total_gm_current = df["Calculated_GM"].sum()

gm_without_top_loss = df[~df["Top_Loss_Flag"]]["Calculated_GM"].sum()

improvement = gm_without_top_loss - total_gm_current

round(total_gm_current, 2), round(gm_without_top_loss, 2), round(improvement, 2)

(np.float64(208045169.25), np.float64(295025530.64), np.float64(86980361.39))

In [31]:
# Revenue contribution of Top Loss SKUs
revenue_top_loss = df[df["Top_Loss_Flag"]]["Revenue"].sum()
revenue_total = df["Revenue"].sum()

round(revenue_top_loss, 2), round((revenue_top_loss / revenue_total) * 100, 2)

(np.float64(51288620.86), np.float64(7.01))

In [32]:
# Transaction intensity check
txn_top_loss = df[df["Top_Loss_Flag"]]["SKU_ID"].count()
txn_total = df["SKU_ID"].count()

round((txn_top_loss / txn_total) * 100, 2)

np.float64(16.99)

In [33]:
# Supplier concentration within Top Loss SKUs

top_loss_supplier = df[df["Top_Loss_Flag"]].groupby("Supplier").agg(
    Transactions=("SKU_ID", "count"),
    Total_Loss=("Calculated_GM", "sum")
)

top_loss_supplier.sort_values("Total_Loss")

,Transactions,Total_Loss
Supplier,,
Allianz Business Corporation,3168,"-19,534,991.89"
Kusum Tradex Private Limited,2430,"-16,560,177.66"
Pegasus Corporation,2595,"-15,556,339.43"
Hariom Enterprises,2044,"-15,205,814.89"
Saluja & co.,1368,"-10,937,456.57"
HINDUSTAN ASSOCIATES,1420,"-9,185,580.95"


In [34]:
# Inventory exposure of Top Loss SKUs
inventory_compare = df.groupby("Top_Loss_Flag").agg(
    Avg_Stock_Level=("Stock_Level", "mean")
)

inventory_compare

,Avg_Stock_Level
Top_Loss_Flag,
False,380.28
True,381.14


In [39]:
# Sanitaryware Baseline Performance

# Filter sanitaryware data
sanitary = df[df["Category"] == "Sanitaryware"]

# Aggregate core financials
sanitary_summary = sanitary.groupby("Category").agg(
    Total_Revenue=("Revenue", "sum"),
    Total_Cost=("Cost", "sum"),
    Total_GM=("Calculated_GM", "sum"),
    Transactions=("SKU_ID", "count")
)

# Calculate margin %
sanitary_summary["GM_%"] = (
    sanitary_summary["Total_GM"] / sanitary_summary["Total_Revenue"]
) * 100

sanitary_summary

,Total_Revenue,Total_Cost,Total_GM,Transactions,GM_%
Category,,,,,
Sanitaryware,"106,444,876.20","77,417,823.27","29,027,052.93",12028,27.27


In [40]:
# Sanitaryware Revenue by Channel

sanitary_channel = sanitary.groupby("Sales_Channel").agg(
    Revenue=("Revenue", "sum"),
    GM=("Calculated_GM", "sum"),
    Transactions=("SKU_ID", "count")
)

sanitary_channel["GM_%"] = (
    sanitary_channel["GM"] / sanitary_channel["Revenue"]
) * 100

sanitary_channel.sort_values("Revenue", ascending=False)

,Revenue,GM,Transactions,GM_%
Sales_Channel,,,,
Retail,"36,287,149.97","10,362,168.15",4045,28.56
Project,"35,100,119.56","9,434,944.75",3988,26.88
Contractor,"35,057,606.67","9,229,940.03",3995,26.33


In [41]:
# Average Revenue per Transaction

sanitary_channel["Avg_Revenue_per_Transaction"] = (
    sanitary_channel["Revenue"] / sanitary_channel["Transactions"]
)

sanitary_channel

,Revenue,GM,Transactions,GM_%,Avg_Revenue_per_Transaction
Sales_Channel,,,,,
Contractor,"35,057,606.67","9,229,940.03",3995,26.33,"8,775.37"
Project,"35,100,119.56","9,434,944.75",3988,26.88,"8,801.43"
Retail,"36,287,149.97","10,362,168.15",4045,28.56,"8,970.87"


In [42]:
# Top Loss SKUs within Sanitaryware

top_loss_sanitary = top_loss_skus[top_loss_skus["Category"] == "Sanitaryware"]

len(top_loss_sanitary)

19

In [43]:
sanitary_top_loss_impact = df[
    (df["Category"] == "Sanitaryware") & 
    (df["Top_Loss_Flag"] == True)
]["Calculated_GM"].sum()

sanitary_total_gm = sanitary["Calculated_GM"].sum()

round(sanitary_top_loss_impact, 2), round(sanitary_total_gm, 2)

(np.float64(-17268424.64), np.float64(29027052.93))

In [44]:
# Sanitaryware Revenue Pareto

sanitary_sku_revenue = sanitary.groupby("SKU_ID").agg(
    Revenue=("Revenue", "sum")
).sort_values("Revenue", ascending=False)

sanitary_sku_revenue["Cumulative_%"] = (
    sanitary_sku_revenue["Revenue"].cumsum() / 
    sanitary_sku_revenue["Revenue"].sum()
) * 100

sanitary_sku_revenue.head(20)

,Revenue,Cumulative_%
SKU_ID,,
SKU00354,"2,269,983.87",2.13
SKU00204,"2,268,620.76",4.26
SKU00263,"2,265,712.18",6.39
SKU00030,"2,254,331.69",8.51
SKU00016,"2,243,005.98",10.62
SKU00408,"2,214,014.81",12.70
SKU00397,"2,184,145.08",14.75
SKU00482,"2,100,991.61",16.72
SKU00577,"2,094,788.80",18.69


In [46]:
# Profitability of Top Revenue SKUs

top_20_skus = sanitary_sku_revenue.head(20).index.tolist()

top_20_df = sanitary[sanitary["SKU_ID"].isin(top_20_skus)]

top_20_revenue = top_20_df["Revenue"].sum()
top_20_gm = top_20_df["Calculated_GM"].sum()

top_20_gm_percent = (top_20_gm / top_20_revenue) * 100

top_20_revenue, top_20_gm, round(top_20_gm_percent, 2)

(np.float64(40212412.900000006),
 np.float64(22308300.509999998),
 np.float64(55.48))

In [47]:
# SKUs contributing 80% of Sanitaryware revenue
sanitary_sku_revenue_reset = sanitary_sku_revenue.reset_index()

sku_80 = sanitary_sku_revenue_reset[
    sanitary_sku_revenue_reset["Cumulative_%"] >= 80
].head(1)

sku_80

,SKU_ID,Revenue,Cumulative_%
52,SKU00008,"1,057,672.00",80.30
